# Module 8.1: Time-Varying Parameters in Dynamic Factor Models

## Learning Objectives

By completing this notebook, you will:

1. Understand why factor loadings and dynamics may change over time in economic data
2. Implement time-varying parameter (TVP) factor models using Kalman filtering
3. Detect structural breaks in factor loadings using the Bai-Perron approach
4. Compare rolling window, state-space TVP, and change-point methods
5. Apply TVP models to data with known regime changes and evaluate forecast performance

## Prerequisites

- State-space representation (Module 2)
- Kalman filter and EM algorithm (Module 4)
- Dynamic factor models

## Estimated Time: 90 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "State-space representation (Module 2)",
    "Kalman filter and EM algorithm (Module 4)",
    "Dynamic factor models"
])

## Setup and Imports

We'll implement time-varying parameter models using Kalman filtering for parameter evolution.

In [ ]:
# Purpose: Import libraries for time-varying parameter models
# Key Concept: Parameters drift or shift over time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy.linalg import lstsq
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("Libraries imported successfully!")

## 1. Motivation for Time-Varying Parameters

### Why Parameters Change

Economic structure evolves due to:
- **Policy regime shifts**: Changes in monetary/fiscal policy
- **Technological innovation**: New industries alter factor structures
- **Globalization**: Increased international co-movement
- **Structural breaks**: Financial crises, recessions

**Example**: The relationship between oil prices and inflation weakened after the 1980s due to:
- Energy efficiency improvements
- Monetary policy credibility
- Structural economic changes

### Model Formulation

**Static parameter model** (restrictive):
$$X_t = \Lambda F_t + e_t$$
$$F_t = \Phi F_{t-1} + \eta_t$$

**Time-varying parameter model** (flexible):
$$X_t = \Lambda_t F_t + e_t$$
$$F_t = \Phi_t F_{t-1} + \eta_t$$

where $\Lambda_t$ and/or $\Phi_t$ evolve over time.

### Generate Data with Structural Break

Let's create data where factor loadings change midway through the sample.

In [ ]:
# Purpose: Generate data with time-varying loadings
# Key Concept: Loadings change at known break point

def generate_tvp_data(T=200, N=10, r=2, break_point=None, change_magnitude=0.5):
    """
    Generate factor model data with structural break in loadings.
    
    Parameters
    ----------
    T : int
        Number of time periods
    N : int
        Number of variables
    r : int
        Number of factors
    break_point : int, optional
        Time of structural break (default: T//2)
    change_magnitude : float
        Size of loading change
    
    Returns
    -------
    X, F_true, Lambda_pre, Lambda_post, break_point
    """
    if break_point is None:
        break_point = T // 2
    
    # Step 1: Generate factors with VAR(1) dynamics
    Phi = np.array([[0.9, 0.1], [0.0, 0.8]])[:r, :r]
    F_true = np.zeros((T, r))
    F_true[0] = np.random.randn(r)
    
    for t in range(1, T):
        F_true[t] = Phi @ F_true[t-1] + np.random.randn(r) * 0.3
    
    # Step 2: Create pre-break and post-break loadings
    Lambda_pre = np.random.randn(N, r)
    Lambda_post = Lambda_pre + np.random.randn(N, r) * change_magnitude
    
    # Step 3: Generate X with smooth transition in loadings
    X = np.zeros((T, N))
    
    for t in range(T):
        if t < break_point:
            # Pre-break period
            Lambda_t = Lambda_pre
        else:
            # Post-break: smooth transition over 20 periods
            transition_length = 20
            if t < break_point + transition_length:
                weight = (t - break_point) / transition_length
                Lambda_t = (1 - weight) * Lambda_pre + weight * Lambda_post
            else:
                Lambda_t = Lambda_post
        
        X[t] = Lambda_t @ F_true[t] + np.random.randn(N) * 0.3
    
    return X, F_true, Lambda_pre, Lambda_post, break_point

# Generate data
T, N, r = 200, 10, 2
X, F_true, Lambda_pre, Lambda_post, break_point = generate_tvp_data(T=T, N=N, r=r)

print("="*70)
print("DATA WITH TIME-VARYING LOADINGS")
print("="*70)
print(f"Time periods (T): {T}")
print(f"Variables (N): {N}")
print(f"Factors (r): {r}")
print(f"Structural break at t = {break_point}")
print(f"\nLoading change for Variable 0, Factor 0:")
print(f"  Pre-break: {Lambda_pre[0, 0]:.3f}")
print(f"  Post-break: {Lambda_post[0, 0]:.3f}")
print(f"  Change: {Lambda_post[0, 0] - Lambda_pre[0, 0]:.3f}")
print("="*70)

### Visualize the Data Structure

In [ ]:
# Purpose: Visualize time-varying structure
# Key Concept: Data characteristics change at break point

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: True factors
axes[0, 0].plot(F_true[:, 0], label='Factor 1', linewidth=2)
axes[0, 0].plot(F_true[:, 1], label='Factor 2', linewidth=2)
axes[0, 0].axvline(break_point, color='red', linestyle='--', 
                   linewidth=2, label='Break point')
axes[0, 0].set_xlabel('Time', fontsize=11)
axes[0, 0].set_ylabel('Factor Value', fontsize=11)
axes[0, 0].set_title('True Factors', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Sample variables showing regime change
for i in range(3):
    axes[0, 1].plot(X[:, i], label=f'Variable {i}', linewidth=1.5, alpha=0.7)
axes[0, 1].axvline(break_point, color='red', linestyle='--', 
                   linewidth=2, label='Break point')
axes[0, 1].set_xlabel('Time', fontsize=11)
axes[0, 1].set_ylabel('Value', fontsize=11)
axes[0, 1].set_title('Observed Variables', fontsize=12, fontweight='bold')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Loading comparison (first variable)
var_idx = 0
x_pos = np.arange(r)
width = 0.35
axes[1, 0].bar(x_pos - width/2, Lambda_pre[var_idx, :], width, 
              label='Pre-break', alpha=0.7, edgecolor='black')
axes[1, 0].bar(x_pos + width/2, Lambda_post[var_idx, :], width, 
              label='Post-break', alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Factor', fontsize=11)
axes[1, 0].set_ylabel('Loading', fontsize=11)
axes[1, 0].set_title(f'Loading Change: Variable {var_idx}', 
                     fontsize=12, fontweight='bold')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels([f'F{i+1}' for i in range(r)])
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Rolling variance showing structural change
window = 20
rolling_var = pd.DataFrame(X).rolling(window=window).var().mean(axis=1)
axes[1, 1].plot(rolling_var, linewidth=2, color='steelblue')
axes[1, 1].axvline(break_point, color='red', linestyle='--', 
                   linewidth=2, label='Break point')
axes[1, 1].set_xlabel('Time', fontsize=11)
axes[1, 1].set_ylabel('Average Variance', fontsize=11)
axes[1, 1].set_title(f'Rolling Variance (window={window})', 
                     fontsize=12, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Time-Varying Parameter Factor Model

### State-Space Formulation

We augment the state vector to include time-varying loadings:

**State equation**:
$$\begin{bmatrix} F_t \\ \text{vec}(\Lambda_t) \end{bmatrix} = \begin{bmatrix} \Phi & 0 \\ 0 & I \end{bmatrix} \begin{bmatrix} F_{t-1} \\ \text{vec}(\Lambda_{t-1}) \end{bmatrix} + \begin{bmatrix} \eta_t \\ \nu_t \end{bmatrix}$$

**Observation equation**:
$$X_t = \Lambda_t F_t + e_t$$

**Challenge**: This is **nonlinear** in the augmented state! We need extended Kalman filter or simplifications.

In [ ]:
# Purpose: Implement TVP-DFM with time-varying loadings
# Key Concept: Kalman filter tracks both factors and loadings

class TVPDynamicFactorModel:
    """
    Time-Varying Parameter Dynamic Factor Model.
    
    Implements random walk evolution for factor loadings:
        Lambda_t = Lambda_{t-1} + nu_t
    
    Parameters
    ----------
    n_factors : int
        Number of latent factors
    Q_scale : float
        Variance of loading innovations (controls smoothness)
    """
    
    def __init__(self, n_factors, Q_scale=0.01):
        self.n_factors = n_factors
        self.Q_scale = Q_scale
        
        self.Lambda_t = None  # Time-varying loadings (N x r x T)
        self.Phi = None       # Factor dynamics
        self.F_t = None       # Estimated factors
    
    def fit(self, X, n_iter=5, verbose=True):
        """
        Estimate TVP-DFM using iterative Kalman filtering.
        
        Parameters
        ----------
        X : array, shape (T, N)
            Observed data
        n_iter : int
            Number of EM iterations
        verbose : bool
            Print progress
        """
        T, N = X.shape
        r = self.n_factors
        
        # Step 1: Initialize with static PCA
        pca = PCA(n_components=r)
        F_init = pca.fit_transform(X)
        Lambda_init = pca.components_.T
        
        # Initialize storage for time-varying loadings
        self.Lambda_t = np.tile(Lambda_init[:, :, np.newaxis], (1, 1, T))
        self.F_t = F_init.copy()
        
        # Estimate initial factor dynamics
        self.Phi = self._estimate_var_dynamics(self.F_t)
        
        # Main EM loop
        for iteration in range(n_iter):
            if verbose:
                print(f"Iteration {iteration + 1}/{n_iter}...")
            
            # E-step: Filter factors given current Lambda_t
            self.F_t = self._kalman_filter_factors(X)
            
            # M-step: Update time-varying loadings
            self._update_tvp_loadings(X)
            
            # M-step: Update dynamics
            self.Phi = self._estimate_var_dynamics(self.F_t)
        
        if verbose:
            print("TVP-DFM estimation complete!")
        
        return self
    
    def _kalman_filter_factors(self, X):
        """
        Kalman filter for factors with time-varying loadings.
        """
        T, N = X.shape
        r = self.n_factors
        
        # Initialize
        F_filt = np.zeros((T, r))
        P_filt = np.zeros((T, r, r))
        
        # Initial condition
        F_filt[0] = np.linalg.lstsq(self.Lambda_t[:, :, 0], X[0], rcond=None)[0]
        P_filt[0] = np.eye(r)
        
        Q = np.eye(r) * 0.1  # Factor innovation variance
        R = np.eye(N) * 0.5  # Measurement error variance
        
        # Forward pass
        for t in range(1, T):
            # Prediction
            F_pred = self.Phi @ F_filt[t-1]
            P_pred = self.Phi @ P_filt[t-1] @ self.Phi.T + Q
            
            # Update with time-varying Lambda_t
            Lambda_t = self.Lambda_t[:, :, t]
            
            # Innovation
            v = X[t] - Lambda_t @ F_pred
            S = Lambda_t @ P_pred @ Lambda_t.T + R
            
            # Kalman gain
            try:
                K = P_pred @ Lambda_t.T @ np.linalg.inv(S)
            except:
                K = np.zeros((r, N))
            
            # Update
            F_filt[t] = F_pred + K @ v
            P_filt[t] = (np.eye(r) - K @ Lambda_t) @ P_pred
        
        return F_filt
    
    def _update_tvp_loadings(self, X):
        """
        Update time-varying loadings using Kalman filter.
        
        For each variable i, filter lambda_i as state with random walk evolution.
        """
        T, N = X.shape
        r = self.n_factors
        
        Q_lambda = np.eye(r) * self.Q_scale
        
        for i in range(N):
            # Filter loading vector for variable i
            lambda_i_filt = np.zeros((T, r))
            P_lambda = np.zeros((T, r, r))
            
            # Initial condition
            lambda_i_filt[0] = self.Lambda_t[i, :, 0]
            P_lambda[0] = np.eye(r) * 0.1
            
            for t in range(1, T):
                # Prediction (random walk)
                lambda_i_pred = lambda_i_filt[t-1]
                P_pred = P_lambda[t-1] + Q_lambda
                
                # Update using X_{i,t} and F_t
                F_t = self.F_t[t]
                y_t = X[t, i]
                
                # Innovation
                v = y_t - lambda_i_pred @ F_t
                S = F_t.T @ P_pred @ F_t + 0.5
                
                # Kalman gain
                K = (P_pred @ F_t) / (S + 1e-10)
                
                # Update
                lambda_i_filt[t] = lambda_i_pred + K * v
                P_lambda[t] = P_pred - np.outer(K, K) * S
            
            # Store updated loadings
            self.Lambda_t[i, :, :] = lambda_i_filt.T
    
    def _estimate_var_dynamics(self, F):
        """
        Estimate VAR(1) dynamics for factors.
        """
        T, r = F.shape
        Y = F[1:, :]
        X = F[:-1, :]
        
        Phi = lstsq(X, Y, rcond=None)[0].T
        return Phi
    
    def get_loadings(self, t):
        """
        Get loadings at time t.
        
        Parameters
        ----------
        t : int
            Time index
        
        Returns
        -------
        Lambda_t : array, shape (N, r)
        """
        return self.Lambda_t[:, :, t]
    
    def plot_loading_evolution(self, var_idx, factor_idx, true_pre=None, 
                               true_post=None, break_point=None):
        """
        Plot evolution of a specific loading over time.
        """
        T = self.Lambda_t.shape[2]
        loading_path = self.Lambda_t[var_idx, factor_idx, :]
        
        fig, ax = plt.subplots(figsize=(10, 5))
        
        ax.plot(loading_path, linewidth=2, label='Estimated', color='steelblue')
        
        # Plot true loadings if provided
        if true_pre is not None and break_point is not None:
            ax.axhline(true_pre, color='red', linestyle='--', 
                      linewidth=2, alpha=0.5, label='True (pre-break)')
        if true_post is not None and break_point is not None:
            ax.axhline(true_post, color='darkred', linestyle='--', 
                      linewidth=2, alpha=0.5, label='True (post-break)')
        
        # Mark break point
        if break_point is not None:
            ax.axvline(break_point, color='orange', linestyle=':', 
                      linewidth=2, label='Break point')
        
        ax.axhline(0, color='black', linestyle='-', alpha=0.3, linewidth=0.8)
        ax.set_xlabel('Time', fontsize=12)
        ax.set_ylabel('Loading Value', fontsize=12)
        ax.set_title(f'Time-Varying Loading: Variable {var_idx}, Factor {factor_idx}', 
                    fontsize=13, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig

print("TVPDynamicFactorModel class implemented!")

### Fit TVP Model to Data

In [ ]:
# Purpose: Fit TVP-DFM to data with structural break
# Key Concept: Model adapts loadings over time

print("Fitting TVP-DFM with time-varying loadings...")
print("="*70)

model_tvp = TVPDynamicFactorModel(n_factors=r, Q_scale=0.01)
model_tvp.fit(X, n_iter=5, verbose=True)

print("\n" + "="*70)
print("TVP-DFM ESTIMATION RESULTS")
print("="*70)
print(f"Estimated factor dynamics (Phi):")
print(model_tvp.Phi)
print("\nTrue factor dynamics:")
print(np.array([[0.9, 0.1], [0.0, 0.8]]))
print("="*70)

### Visualize Loading Evolution

In [ ]:
# Purpose: Visualize how loadings evolve over time
# Key Concept: Loadings should adapt to structural break

# Plot loading evolution for first variable, first factor
var_idx, factor_idx = 0, 0

fig = model_tvp.plot_loading_evolution(
    var_idx, 
    factor_idx, 
    true_pre=Lambda_pre[var_idx, factor_idx],
    true_post=Lambda_post[var_idx, factor_idx],
    break_point=break_point
)
plt.show()

# Compare estimated loadings at different times
t_pre = 50
t_post = 150

print("\n" + "="*70)
print("LOADING ESTIMATES AT DIFFERENT TIME POINTS")
print("="*70)
print(f"Variable {var_idx}, Factor {factor_idx}:")
print(f"\nTrue pre-break:  {Lambda_pre[var_idx, factor_idx]:.3f}")
print(f"Estimated at t={t_pre}:  {model_tvp.get_loadings(t_pre)[var_idx, factor_idx]:.3f}")
print(f"\nTrue post-break: {Lambda_post[var_idx, factor_idx]:.3f}")
print(f"Estimated at t={t_post}: {model_tvp.get_loadings(t_post)[var_idx, factor_idx]:.3f}")
print("="*70)

## 3. Structural Break Detection

### Bai-Perron Approach

Instead of smooth TVP, we can test for **discrete breaks** in loadings.

**Algorithm**:
1. For each candidate break date $\tau$:
   - Estimate loadings before $\tau$ and after $\tau$
   - Compute sum of squared residuals
2. Choose break minimizing SSR
3. Test if improvement is statistically significant

In [ ]:
# Purpose: Implement structural break detection
# Key Concept: Find discrete regime changes in loadings

def detect_loading_break(X, F, min_segment=40):
    """
    Detect single structural break in factor loadings.
    
    Parameters
    ----------
    X : array, shape (T, N)
        Observed data
    F : array, shape (T, r)
        Estimated factors
    min_segment : int
        Minimum observations in each regime
    
    Returns
    -------
    best_break : int
        Estimated break date
    ssr_no_break : float
        SSR under no break
    ssr_with_break : float
        SSR with break
    Lambda_pre, Lambda_post : arrays
        Estimated loadings in each regime
    """
    T, N = X.shape
    r = F.shape[1]
    
    def estimate_loadings(X_seg, F_seg):
        """Estimate loadings by OLS."""
        Lambda = lstsq(F_seg, X_seg, rcond=None)[0].T
        return Lambda
    
    def compute_ssr(X_seg, F_seg, Lambda):
        """Compute sum of squared residuals."""
        resid = X_seg - F_seg @ Lambda.T
        return np.sum(resid ** 2)
    
    # SSR with no break
    Lambda_full = estimate_loadings(X, F)
    ssr_no_break = compute_ssr(X, F, Lambda_full)
    
    # Search for break
    best_break = None
    best_ssr = np.inf
    best_lambdas = (None, None)
    
    for tau in range(min_segment, T - min_segment):
        # Split data
        X_pre, X_post = X[:tau], X[tau:]
        F_pre, F_post = F[:tau], F[tau:]
        
        # Estimate loadings in each regime
        Lambda_pre = estimate_loadings(X_pre, F_pre)
        Lambda_post = estimate_loadings(X_post, F_post)
        
        # Compute total SSR
        ssr_pre = compute_ssr(X_pre, F_pre, Lambda_pre)
        ssr_post = compute_ssr(X_post, F_post, Lambda_post)
        ssr_total = ssr_pre + ssr_post
        
        if ssr_total < best_ssr:
            best_ssr = ssr_total
            best_break = tau
            best_lambdas = (Lambda_pre, Lambda_post)
    
    return best_break, ssr_no_break, best_ssr, best_lambdas[0], best_lambdas[1]

# Detect break using estimated factors from static PCA
print("Detecting structural break in loadings...")
pca = PCA(n_components=r)
F_static = pca.fit_transform(X)

detected_break, ssr_no_break, ssr_with_break, Lambda_est_pre, Lambda_est_post = \
    detect_loading_break(X, F_static, min_segment=40)

# Compute improvement
improvement = (ssr_no_break - ssr_with_break) / ssr_no_break * 100

print("\n" + "="*70)
print("STRUCTURAL BREAK DETECTION RESULTS")
print("="*70)
print(f"True break point: t = {break_point}")
print(f"Detected break point: t = {detected_break}")
print(f"Detection error: {abs(detected_break - break_point)} periods")
print(f"\nSSR with no break: {ssr_no_break:.2f}")
print(f"SSR with break: {ssr_with_break:.2f}")
print(f"SSR improvement: {improvement:.2f}%")
print("="*70)

# Compare estimated loadings to truth
var_idx = 0
print(f"\nVariable {var_idx} loading estimates:")
print(f"\nFactor 0:")
print(f"  True pre-break:   {Lambda_pre[var_idx, 0]:.3f}")
print(f"  Est. pre-break:   {Lambda_est_pre[var_idx, 0]:.3f}")
print(f"  True post-break:  {Lambda_post[var_idx, 0]:.3f}")
print(f"  Est. post-break:  {Lambda_est_post[var_idx, 0]:.3f}")

## 4. Comparing Methods

Let's compare three approaches:
1. **Static model** (ignores parameter change)
2. **TVP model** (smooth random walk)
3. **Break model** (discrete regime change)

In [ ]:
# Purpose: Compare static, TVP, and break models
# Key Concept: Appropriate model depends on change type

# Generate test data for out-of-sample evaluation
T_test = 50
X_test = np.zeros((T_test, N))
for t in range(T_test):
    F_new = model_tvp.Phi @ F_true[-1] + np.random.randn(r) * 0.3
    X_test[t] = Lambda_post @ F_new + np.random.randn(N) * 0.3
    F_true = np.vstack([F_true, F_new])

# 1. Static model (full sample)
pca_static = PCA(n_components=r)
pca_static.fit(X)
Lambda_static = pca_static.components_.T
mse_static = np.mean((X_test - pca_static.transform(X_test) @ Lambda_static.T) ** 2)

# 2. TVP model (uses final loadings)
Lambda_tvp_final = model_tvp.get_loadings(T-1)
F_test_tvp = X_test @ np.linalg.pinv(Lambda_tvp_final.T)
X_recon_tvp = F_test_tvp @ Lambda_tvp_final.T
mse_tvp = np.mean((X_test - X_recon_tvp) ** 2)

# 3. Break model (post-break loadings)
F_test_break = X_test @ np.linalg.pinv(Lambda_est_post.T)
X_recon_break = F_test_break @ Lambda_est_post.T
mse_break = np.mean((X_test - X_recon_break) ** 2)

# Summary
results_df = pd.DataFrame({
    'Model': ['Static', 'TVP (Random Walk)', 'Break (Discrete)'],
    'Test MSE': [mse_static, mse_tvp, mse_break],
    'Relative MSE': [mse_static/mse_static, mse_tvp/mse_static, mse_break/mse_static]
})

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(results_df.to_string(index=False))
print("\nNote: Test data generated from post-break regime.")
print("="*70)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
x_pos = np.arange(len(results_df))
bars = ax.bar(x_pos, results_df['Test MSE'], 
             color=['coral', 'seagreen', 'steelblue'],
             alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels(results_df['Model'], fontsize=11)
ax.set_ylabel('Test MSE', fontsize=12)
ax.set_title('Out-of-Sample Performance Comparison', 
            fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Highlight best
best_idx = results_df['Test MSE'].argmin()
bars[best_idx].set_edgecolor('red')
bars[best_idx].set_linewidth(3)

plt.tight_layout()
plt.show()

## 5. Exercises

### Exercise 5.1: Varying the Smoothness Parameter

**Task:** Fit TVP models with different `Q_scale` values (0.001, 0.01, 0.1) and analyze how smoothness affects loading paths.

In [ ]:
# Exercise 5.1: Effect of smoothness parameter

Q_scales = [0.001, 0.01, 0.1]

# YOUR CODE HERE
# ---------------
# For each Q_scale:
#   1. Fit TVP model
#   2. Extract loading path for Variable 0, Factor 0
#   3. Compute variance of loading changes

# Plot loading paths for different Q_scales on same figure

# AUTO-GRADED TESTS
def test_exercise_5_1():
    # Test that you've fitted models
    print("Implement Exercise 5.1: Fit TVP models with different Q_scales")
    print("Expected: Plot showing loading paths for Q_scales = [0.001, 0.01, 0.1]")

test_exercise_5_1()

### Exercise 5.2: Rolling Window Estimation

**Task:** Implement rolling window estimation with window size 60 and compare loading estimates to TVP model.

In [ ]:
# Exercise 5.2: Rolling window estimation

window_size = 60

# YOUR CODE HERE
# ---------------
# For each time t >= window_size:
#   1. Extract window: X[t-window_size:t]
#   2. Fit PCA on window
#   3. Store loadings

# Plot comparison: Rolling window vs TVP loadings

# AUTO-GRADED TESTS
def test_exercise_5_2():
    print("Implement Exercise 5.2: Rolling window estimation")
    print("Expected: Comparison plot of rolling window vs TVP loadings")

test_exercise_5_2()

### Exercise 5.3: Multiple Break Detection

**Task:** Generate data with TWO structural breaks and extend the break detection algorithm to find both breaks.

In [ ]:
# Exercise 5.3: Multiple break detection

# Step 1: Generate data with two breaks
# YOUR CODE: Modify generate_tvp_data to include two breaks

# Step 2: Implement sequential break detection
# YOUR CODE: 
#   - Find first break
#   - Split sample at first break
#   - Find second break in one of the segments

# Step 3: Visualize detected breaks vs true breaks

# AUTO-GRADED TESTS
def test_exercise_5_3():
    print("Implement Exercise 5.3: Detect multiple structural breaks")
    print("Expected: Data with 2 breaks and detection algorithm finding both")

test_exercise_5_3()

## 6. Summary

### Key Takeaways

1. **Economic structure evolves** over time due to policy changes, technology, and structural shifts

2. **TVP factor models** allow loadings and/or dynamics to drift, capturing smooth parameter evolution

3. **Kalman filtering** tracks time-varying parameters by treating them as additional state variables

4. **Smoothness parameter Q** controls how quickly parameters can change (bias-variance tradeoff)

5. **Structural break detection** is appropriate when changes are discrete rather than smooth

6. **Model choice matters**: Use TVP for gradual change, break models for abrupt shifts

### Practical Considerations

**When to use TVP:**
- Gradual structural evolution
- No clear break dates
- Real-time adaptive forecasting

**When to use break detection:**
- Discrete regime changes
- Known crisis periods
- Policy regime shifts

**Challenges:**
- High dimensionality (many parameters)
- Identification issues
- Overfitting risk with too much flexibility

### Next Steps

The final notebook explores **machine learning connections** to factor models, comparing autoencoders, neural networks, and traditional PCA.

---

## Additional Resources

**Key Papers:**
- Primiceri, G.E. (2005). "Time varying structural vector autoregressions and monetary policy." *RES*
- Koop, G. & Korobilis, D. (2014). "A new index of financial conditions." *EER*
- Bai, J., Han, X. & Shi, Y. (2020). "Estimation and inference of change points in high-dimensional factor models." *JBES*

**Books:**
- Del Negro, M. & Primiceri, G.E. (2015). "Time varying structural vector autoregressions." *RES*